In [15]:
import pandas as pd
import torch

import ast


In [48]:
# Load final datasets

df_rep = pd.read_csv("../data/processed/final_datasets/final_rep_dataset.csv")
df_relax = pd.read_csv("../data/processed/final_datasets/final_relax_dataset.csv")

negative_rep = pd.read_csv("../data/processed/negative_datasets/neg_dataset_rep.csv")
negative_relax = pd.read_csv("../data/processed/negative_datasets/neg_dataset_relax.csv")

In [49]:
from esm.models.esmc import EsmcForMaskedLM, EsmcTokenizer
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# ESMC-300M Pretrained model
model = EsmcForMaskedLM.from_pretrained(
    "biohub/ESMC-300M",
    device=device,
).eval()

tokenizer = EsmcTokenizer()

cuda


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [50]:
def get_protein_embedding(amino_acid_seq):

    inputs = tokenizer([amino_acid_seq], return_tensors="pt", padding=True)
    inputs = {name: value.to(model.device) for name, value in inputs.items()}

    with torch.inference_mode():
        result = model(**inputs, output_hidden_states=True)

    token_embeddings = result.hidden_states[-1]    
    residue_embeddings = token_embeddings[0, 1:-1] # exlude start and end token

    return residue_embeddings

def get_sDBM_embeds(df):
    sDBM_tensor_list = []

    for index, row in df.iterrows():
        sDBM_start, sDBM_end = ast.literal_eval(row.values[0])
        sDBM_end += 1

        amino_seq = row.values[3]

        sDBM_embedding = get_protein_embedding(amino_seq)[sDBM_start : sDBM_end, :]

    # accumulating embeddings on the gpu -> save them on cpu instead
        sDBM_tensor_list.append(sDBM_embedding.cpu())

    embedding_tensor = torch.stack(sDBM_tensor_list, dim = 0)

    print(embedding_tensor.shape)

    return embedding_tensor


def get_dna_encoding(ori_seq):

    dna_vec_list = []

    for base in ori_seq:
        match base:
            case 'A':
                dna_vec_list.append(0)
            case 'C':
                dna_vec_list.append(1)
            case 'G':
                dna_vec_list.append(2)
            case 'T':  
                dna_vec_list.append(3)

    #convert to tensor
    dna_encoding = torch.tensor(dna_vec_list, dtype=torch.long)

    return dna_encoding


def get_ori_encoding(df):
    ori_tensor_list = []

    for index, row in df.iterrows():

        ori_seq = row.values[2]

        dna_encoding = get_dna_encoding(ori_seq)


    # accumulating embeddings on the gpu -> save them on cpu instead
        ori_tensor_list.append(dna_encoding.unsqueeze(dim=0))


    ori_encoding_tensor = torch.stack(ori_tensor_list, dim = 0)

    ori_encoding = ori_encoding_tensor.flatten(-2)

    return ori_encoding

In [51]:
import ast

from sklearn.preprocessing import StandardScaler
scaleStandard = StandardScaler()
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt

import random

pca1 = PCA()

def apply_pca(embedding, var = 0.30):
    # pca2 = PCA(0.50)
    pca2 = PCA(var)


    embed_np = embedding.cpu().numpy()


    reduced_residue_embeddings = []

    for i in range(0, 9):

        residue_embed = embed_np[:, i, :]

        residue_embed_scaled = scaleStandard.fit_transform(residue_embed)

        residue_embed_pca2 = pca2.fit_transform(residue_embed_scaled)

        reduced_residue_embeddings.append(torch.from_numpy(residue_embed_pca2))

        print(residue_embed_pca2.shape)

    sDBM_embedding = torch.concat(reduced_residue_embeddings, dim = 1)

    print(sDBM_embedding.shape)

    return sDBM_embedding

### Binary Classifer Features

In [52]:
from pathlib import Path
import torch


def build_and_save_tensors(
    df_rep,
    negative_rep,
    df_relax,
    negative_relax,
    get_sDBM_embeds,
    get_ori_encoding,
    apply_PCA,
    save_path,
    train_fraction=0.8,
    seed=52,
):
    def split_df(df, split_seed):
        generator = torch.Generator().manual_seed(split_seed)
        order = torch.randperm(len(df), generator=generator).tolist()
        split_at = int(len(df) * train_fraction)
        return df.iloc[order[:split_at]].copy(), df.iloc[order[split_at:]].copy()

    rep_train, rep_val = split_df(df_rep, seed)
    neg_rep_train, neg_rep_val = split_df(negative_rep, seed + 1)
    relax_train, relax_val = split_df(df_relax, seed + 2)
    neg_relax_train, neg_relax_val = split_df(negative_relax, seed + 3)

    # Keep groups in a fixed order so the reduced tensor can be split back correctly.
    groups = [
        rep_train,
        neg_rep_train,
        relax_train,
        neg_relax_train,
        rep_val,
        neg_rep_val,
        relax_val,
        neg_relax_val,
    ]

    # Concatenate all protein embeddings, apply PCA once, then restore the groups.
    raw_proteins = torch.cat(
        [get_sDBM_embeds(group).detach().cpu() for group in groups],
        dim=0,
    )
    reduced_proteins = apply_PCA(raw_proteins)

    group_sizes = [len(group) for group in groups]
    reduced_groups = torch.split(reduced_proteins, group_sizes, dim=0)

    def add_dna(reduced_protein, df):
        dna = get_ori_encoding(df).detach().cpu().float()
        reduced_protein = reduced_protein.detach().cpu().float()

        if reduced_protein.shape[0] != dna.shape[0]:
            raise ValueError("Protein and DNA embeddings have different row counts.")

        return torch.cat((reduced_protein, dna), dim=1)

    train_groups = groups[:4]
    train_reduced = reduced_groups[:4]

    X_train = torch.cat(
        [add_dna(protein, df) for protein, df in zip(train_reduced, train_groups)],
        dim=0,
    )
    y_train = torch.cat([
        torch.full((len(rep_train),), 1, dtype=torch.long),
        torch.full((len(neg_rep_train),), 0, dtype=torch.long),
        torch.full((len(relax_train),), 1, dtype=torch.long),
        torch.full((len(neg_relax_train),), 0, dtype=torch.long),
    ])

    # Shuffle training features and labels together.
    generator = torch.Generator().manual_seed(seed)
    order = torch.randperm(len(y_train), generator=generator)
    X_train, y_train = X_train[order], y_train[order]

    X_val_rep = torch.cat((
        add_dna(reduced_groups[4], rep_val),
        add_dna(reduced_groups[5], neg_rep_val),
    ))
    y_val_rep = torch.cat((
        torch.ones(len(rep_val), dtype=torch.long),
        torch.zeros(len(neg_rep_val), dtype=torch.long),
    ))

    X_val_relax = torch.cat((
        add_dna(reduced_groups[6], relax_val),
        add_dna(reduced_groups[7], neg_relax_val),
    ))
    y_val_relax = torch.cat((
        torch.ones(len(relax_val), dtype=torch.long),
        torch.zeros(len(neg_relax_val), dtype=torch.long),
    ))

    tensors = {
        "X_train": X_train,
        "y_train": y_train,
        "X_val_rep": X_val_rep,
        "y_val_rep": y_val_rep,
        "X_val_relax": X_val_relax,
        "y_val_relax": y_val_relax,
    }

    save_dir = Path(save_path)
    save_dir.mkdir(parents=True, exist_ok=True)

    for name, tensor in tensors.items():
        torch.save(tensor, save_dir / f"{name}.pt")

    return tensors

In [53]:
tensors = build_and_save_tensors(
    df_rep=df_rep,
    negative_rep=negative_rep,
    df_relax=df_relax,
    negative_relax=negative_relax,
    get_sDBM_embeds=get_sDBM_embeds,
    get_ori_encoding=get_ori_encoding,
    apply_PCA=apply_pca,
    save_path=r"../data/embeddings/classifier_combined/",
)

torch.Size([183, 9, 960])
torch.Size([183, 9, 960])
torch.Size([172, 9, 960])
torch.Size([172, 9, 960])
torch.Size([46, 9, 960])
torch.Size([46, 9, 960])
torch.Size([43, 9, 960])
torch.Size([43, 9, 960])
(888, 2)
(888, 2)
(888, 3)
(888, 3)
(888, 3)
(888, 3)
(888, 3)
(888, 3)
(888, 3)
torch.Size([888, 25])


### Ori Predictor Features

In [37]:
rep_sDBM = get_sDBM_embeds(df_rep)
rep_ori_encod = get_ori_encoding(df_rep)

relax_sDBM = get_sDBM_embeds(df_relax)
relax_ori_encod = get_ori_encoding(df_relax)

torch.Size([229, 9, 960])
torch.Size([215, 9, 960])


In [ ]:
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def fit_pca_transform(X_train, X_val, var=0.30):
    train_np = X_train.detach().cpu().numpy()
    val_np = X_val.detach().cpu().numpy()

    train_reduced = []
    val_reduced = []

    for i in range(9):
        scaler = StandardScaler()

        train_scaled = scaler.fit_transform(train_np[:, i, :])
        val_scaled = scaler.transform(val_np[:, i, :])

        pca = PCA(n_components=var)
        train_pca = pca.fit_transform(train_scaled)
        val_pca = pca.transform(val_scaled)

        train_reduced.append(torch.from_numpy(train_pca))
        val_reduced.append(torch.from_numpy(val_pca))

        print(f"Residue {i}: {train_pca.shape[1]} components")

    X_train_pca = torch.cat(train_reduced, dim=1)
    X_val_pca = torch.cat(val_reduced, dim=1)

    print("Train shape:", X_train_pca.shape)
    print("Validation shape:", X_val_pca.shape)

    return X_train_pca, X_val_pca

In [43]:
# Shuffle each group before splitting
perm_rep = torch.randperm(len(rep_sDBM))
shuffle_X_rep = rep_sDBM[perm_rep]
shuffle_y_rep = rep_ori_encod[perm_rep]

perm_relax = torch.randperm(len(relax_sDBM))
shuffle_X_relax = relax_sDBM[perm_relax]
shuffle_y_relax = relax_ori_encod[perm_relax]

# Split each group
train_size_rep = int(0.8 * len(shuffle_X_rep))
train_size_relax = int(0.8 * len(shuffle_X_relax))

X_train_rep = shuffle_X_rep[:train_size_rep]
y_train_rep = shuffle_y_rep[:train_size_rep]
X_val_rep = shuffle_X_rep[train_size_rep:]
y_val_rep = shuffle_y_rep[train_size_rep:]

X_train_relax = shuffle_X_relax[:train_size_relax]
y_train_relax = shuffle_y_relax[:train_size_relax]
X_val_relax = shuffle_X_relax[train_size_relax:]
y_val_relax = shuffle_y_relax[train_size_relax:]

# Combine the raw embeddings and labels
X_train_raw = torch.cat([X_train_rep, X_train_relax], dim=0)
y_train = torch.cat([y_train_rep, y_train_relax], dim=0)

X_val_raw = torch.cat([X_val_rep, X_val_relax], dim=0)
y_val = torch.cat([y_val_rep, y_val_relax], dim=0)

# Shuffle training rows and labels together
perm = torch.randperm(len(X_train_raw))
X_train_raw = X_train_raw[perm]
y_train = y_train[perm]

# Fit PCA on training embeddings; transform both sets
X_train, X_val = fit_pca_transform(X_train_raw, X_val_raw, var=0.30)

Residue 0: 2 components
Residue 1: 2 components
Residue 2: 3 components
Residue 3: 2 components
Residue 4: 3 components
Residue 5: 3 components
Residue 6: 3 components
Residue 7: 2 components
Residue 8: 3 components
Train shape: torch.Size([355, 23])
Validation shape: torch.Size([89, 23])


In [44]:
torch.save(X_train, "../data/embeddings/ori_predictor_combined/combined_X_train.pt")
torch.save(y_train, "../data/embeddings/ori_predictor_combined/combined_y_train.pt")

torch.save(X_val_rep, "../data/embeddings/ori_predictor_combined/X_val_rep.pt")
torch.save(X_val_relax, "../data/embeddings/ori_predictor_combined/X_val_relax.pt")

torch.save(y_val_rep, "../data/embeddings/ori_predictor_combined/y_val_rep.pt")
torch.save(y_val_relax, "../data/embeddings/ori_predictor_combined/y_val_relax.pt")